In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
len(documents)

72

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


In [8]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

for r in results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [9]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

In [10]:
def llm(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response

In [11]:
def search(query):
    return index.search(
        query=query,
        num_results=5
    )


def build_prompt(question, search_results):
    context = ""

    for doc in search_results:
        context += f"Filename: {doc['filename']}\n"
        context += f"Content:\n{doc['content']}\n\n"

    prompt = f"""
You are a course teaching assistant.

Answer the question based only on the context below.
If the answer is not in the context, say "I don't know."

Question:
{question}

Context:
{context}
""".strip()

    return prompt


def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    response = llm(prompt)

    answer = response.content[0].text
    usage = response.usage

    return answer, usage

In [12]:
question = "How does the agentic loop keep calling the model until it stops?"

answer, usage = rag(question)

print(answer)
print(usage)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)

Based on the context provided, the agentic loop keeps calling the model until it stops by using a **`has_function_calls` flag** that tracks whether the model's response contains any function calls.

Here's how it works:

1. **The flag is set to `False` at the start of each iteration**
2. **During response processing**, if any item in the response has `type == "function_call"`, the flag is set to `True`
3. **At the end of each iteration**, the loop checks: `if has_function_calls == False: break`
4. **If there are no function calls** (meaning the model returned only a final message/answer), the loop exits
5. **If there are function calls**, the loop continues, allowing the model to see the tool results and decide what to do next

As stated in the context: *"The exit condition is the simplest one possible. No function calls this turn means we're done."*

The model itself decides when to stop by choosing not to call any more tools and instead returning a final answer message.
Usage(cache_c

In [13]:
usage.input_tokens

8202

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [15]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [16]:
def search_chunks(query):
    return chunk_index.search(
        query=query,
        num_results=5
    )


def rag_chunked(question):
    search_results = search_chunks(question)
    prompt = build_prompt(question, search_results)
    response = llm(prompt)

    answer = response.content[0].text
    usage = response.usage

    return answer, usage

In [17]:
chunked_answer, chunked_usage = rag_chunked(question)

print(chunked_answer)
print("Chunked input tokens:", chunked_usage.input_tokens)

Based on the context provided, the agentic loop keeps calling the model until it stops by using a **flag-based exit condition**. Here's how it works:

The loop checks a `has_function_calls` flag on each iteration. This flag is set to `True` whenever the model's response includes a function call, and remains `False` if there are no function calls.

**The exit condition is**: No function calls this turn means we're done.

Specifically, the code:
1. Sets `has_function_calls = False` at the start of each iteration
2. Calls the model and processes the response
3. If any item in the response is a function call, sets `has_function_calls = True`
4. At the end of the iteration, checks: `if has_function_calls == False: break`

As stated in the context: "The loop keeps calling the model until it returns a response without any function calls."

The context also mentions that "Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit" to prevent infi

In [18]:
print("Original input tokens:", usage.input_tokens)
print("Chunked input tokens:", chunked_usage.input_tokens)

reduction = usage.input_tokens / chunked_usage.input_tokens
print("Reduction factor:", reduction)

Original input tokens: 8202
Chunked input tokens: 2658
Reduction factor: 3.0857787810383748


In [19]:
import json

search_call_count = 0


def search_tool(query: str):
    global search_call_count
    search_call_count += 1

    results = search_chunks(query)

    return [
        {
            "filename": r["filename"],
            "content": r["content"][:1500]
        }
        for r in results
    ]

In [20]:
tools = [
    {
        "name": "search",
        "description": "Search the course lesson documents for relevant information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                }
            },
            "required": ["query"]
        }
    }
]

In [21]:
def agent(question, max_turns=10):
    global search_call_count
    search_call_count = 0

    messages = [
        {
            "role": "user",
            "content": f"""
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.

Question:
{question}
""".strip()
        }
    ]

    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            tools=tools,
            messages=messages
        )

        messages.append(
            {
                "role": "assistant",
                "content": response.content
            }
        )

        tool_results = []

        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input

                if tool_name == "search":
                    result = search_tool(tool_input["query"])

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        }
                    )

        if not tool_results:
            return response.content[0].text, search_call_count

        messages.append(
            {
                "role": "user",
                "content": tool_results
            }
        )

    return "Max turns reached", search_call_count

In [22]:
agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

agent_answer, num_search_calls = agent(agent_question)

print(agent_answer)
print("Search calls:", num_search_calls)

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 10,000 input tokens per minute (org: 541cb09b-3b47-4e23-9701-147b2c7d8d80, model: claude-sonnet-4-5-20250929). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Reduce the prompt length or the maximum tokens requested, or try again later. View your current limits at https://console.anthropic.com/settings/limits. You may also contact sales at https://claude.com/contact-sales to discuss your options for a rate limit increase."}, 'request_id': 'req_011CcBFuCGzBbBAT8EYhKcnr'}